# Parkinson Classifier from PPMI SPECT (DaT) - Slice-Level

This notebook trains a binary classifier using SPECT DICOM images from PPMI.

Label mapping:
- PD -> 1
- Prodromal -> 0
- Control -> 0

Pipeline:
1. Load CSV metadata
2. Map subjects to SPECT folders
3. Build slice-level samples from central DICOM slices per subject
4. Preprocess to 3x224x224
5. Train/validation split
6. Train ResNet18 for binary classification
7. Report validation accuracy

In [1]:
from pathlib import Path
import random
from typing import Dict, List, Optional, Tuple
import subprocess
import sys
from copy import deepcopy

import numpy as np
import pandas as pd

SEED = 42 # Define SEED for reproducibility

In [2]:
%pip install pydicom

In [3]:
import os
import glob
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TVF
from torchvision.transforms import InterpolationMode

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Resolve project paths
# Assuming current working directory is /content/ in Colab
PROJECT_ROOT = Path.cwd().resolve()

CSV_PATH = PROJECT_ROOT / "spect_4_10_2026.csv"
SPECT_ROOT = Path("/content/drive/MyDrive/spect") # User specified absolute path

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not find CSV file: {CSV_PATH}")

if not SPECT_ROOT.exists():
    raise FileNotFoundError(f"Could not find SPECT folder: {SPECT_ROOT}. Please ensure it is mounted and accessible.")

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV exists: {CSV_PATH.exists()} -> {CSV_PATH}")
print(f"SPECT root exists: {SPECT_ROOT.exists()} -> {SPECT_ROOT}")

# 1) Load CSV and keep Subject/Group
meta = pd.read_csv(CSV_PATH)
subject_col = "Subject" if "Subject" in meta.columns else "subject"
group_col = "Group" if "Group" in meta.columns else "group"

meta = meta[[subject_col, group_col]].copy()
meta.columns = ["Subject", "Group"]

label_map = {
    "PD": 1,
    "Prodromal": 0,
    "Control": 0,
}

meta["Group"] = meta["Group"].astype(str).str.strip()
meta = meta[meta["Group"].isin(label_map.keys())].dropna(subset=["Subject", "Group"]).reset_index(drop=True)
meta["label"] = meta["Group"].map(label_map).astype(int)

print("Rows after filtering:", len(meta))
print(meta["Group"].value_counts())
meta.head()

Project root: /content
CSV exists: True -> /content/spect_4_10_2026.csv
SPECT root exists: True -> /content/drive/MyDrive/spect
Rows after filtering: 24
Group
PD         23
Control     1
Name: count, dtype: int64


,Subject,Group,label
0,100018,PD,1
1,100017,PD,1
2,100017,PD,1
3,100017,PD,1
4,100012,PD,1


In [6]:
def normalize_subject_id(value: object) -> Optional[str]:
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits:
        return str(int(digits))
    return s.lower() if s else None


def leading_digits(text: str) -> str:
    out = []
    for ch in text:
        if ch.isdigit():
            out.append(ch)
        else:
            break
    return "".join(out)


def build_subject_folder_index(root: Path) -> Dict[str, Path]:
    index: Dict[str, Path] = {}
    for subdir in root.iterdir():
        if not subdir.is_dir():
            continue
        name = subdir.name
        digits_all = "".join(ch for ch in name if ch.isdigit())
        digits_lead = leading_digits(name)
        keys = {name.lower()}
        for d in [digits_all, digits_lead]:
            if d:
                keys.add(d)
                keys.add(str(int(d)))
        for k in keys:
            index.setdefault(k, subdir)
    return index


def find_dicom_files(subject_folder: Path) -> List[Path]:
    files = list(subject_folder.rglob("*.dcm")) + list(subject_folder.rglob("*.DCM"))
    return sorted(set(files))


def _slice_sort_key(ds: pydicom.Dataset, fallback: int) -> float:
    if hasattr(ds, "InstanceNumber"):
        try:
            return float(ds.InstanceNumber)
        except Exception:
            pass
    if hasattr(ds, "ImagePositionPatient"):
        try:
            return float(ds.ImagePositionPatient[2])
        except Exception:
            pass
    return float(fallback)


def select_central_indices(n: int, k: int = 7) -> List[int]:
    if n <= 0:
        return []
    k = max(1, min(k, n))
    center = n // 2
    start = center - (k // 2)
    if start < 0:
        start = 0
    end = start + k
    if end > n:
        end = n
        start = end - k
    return list(range(start, end))


def collect_central_dicom_slice_files(subject_folder: Path, num_central_slices: int = 7) -> List[Path]:
    dicom_paths = find_dicom_files(subject_folder)
    if not dicom_paths:
        return []

    keyed: List[Tuple[float, Path]] = []
    for i, p in enumerate(dicom_paths):
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
            keyed.append((_slice_sort_key(ds, i), p))
        except Exception:
            continue

    if not keyed:
        return []

    keyed.sort(key=lambda x: x[0])
    ordered_paths = [x[1] for x in keyed]
    idxs = select_central_indices(len(ordered_paths), k=num_central_slices)
    return [ordered_paths[i] for i in idxs]


def load_dicom_slice_from_file(dcm_path: Path) -> Optional[np.ndarray]:
    try:
        ds = pydicom.dcmread(dcm_path)
        arr = ds.pixel_array.astype(np.float32)
        if arr.ndim != 2:
            return None
        if not np.isfinite(arr).all():
            return None
        if float(arr.max()) <= float(arr.min()):
            return None
        return arr
    except Exception:
        return None


def preprocess_to_3ch_tensor(img_2d: np.ndarray, size: int = 224) -> torch.Tensor:
    img = img_2d.astype(np.float32)
    min_v, max_v = float(img.min()), float(img.max())
    if max_v > min_v:
        img = (img - min_v) / (max_v - min_v)
    else:
        img = np.zeros_like(img, dtype=np.float32)

    x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # [1,1,H,W]
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    x = x.squeeze(0).repeat(3, 1, 1)  # [3,224,224]
    return x

In [9]:
print(meta["Group"].value_counts())

print()

# Re-defining subject_df here to resolve NameError, assuming previous cells defining
# functions and SPECT_ROOT have been executed. This ensures 'subject_df' is available
# if the kernel state was reset or the previous cell was not executed in the current session.
subject_index = build_subject_folder_index(SPECT_ROOT)
temp_meta = meta.copy() # Use a copy to avoid modifying the global 'meta' if it's already processed
temp_meta["subject_key"] = temp_meta["Subject"].apply(normalize_subject_id)
temp_meta["subject_folder"] = temp_meta["subject_key"].apply(lambda x: subject_index.get(x) if x is not None else None)
valid_meta = temp_meta[temp_meta["subject_key"].notna() & temp_meta["subject_folder"].notna()].copy()

if len(valid_meta) > 0:
    subject_df = (
        valid_meta.sort_values(["subject_key", "Subject"])
        .groupby("subject_key", as_index=False)
        .agg({
            "Subject": "first",
            "label": lambda x: int(pd.Series(x).mode().iloc[0]),
            "subject_folder": "first",
        })
    )
    print(subject_df[["Subject","label"]])
else:
    print("No valid subjects found to create 'subject_df'.")

Group
PD         23
Control     1
Name: count, dtype: int64

   Subject  label
0   100001      1
1   100002      1
2   100004      0
3   100005      1
4   100006      1
5   100007      1
6   100012      1
7   100017      1
8   100018      1


In [20]:
# Subject-level 3-fold cross validation pipeline (no leakage)
subject_index = build_subject_folder_index(SPECT_ROOT)

# 1) Start from CSV-derived metadata and extract unique Subject IDs
meta["subject_key"] = meta["Subject"].apply(normalize_subject_id)
meta["subject_folder"] = meta["subject_key"].apply(lambda x: subject_index.get(x) if x is not None else None)

mapped_rows = meta["subject_folder"].notna().sum()
print(f"Mapped subject rows: {mapped_rows}/{len(meta)}")

valid_meta = meta[meta["subject_key"].notna() & meta["subject_folder"].notna()].copy()
if len(valid_meta) == 0:
    raise RuntimeError("No valid subjects were mapped to SPECT folders.")

# One row per subject key to enforce subject-level separation
subject_df = (
    valid_meta.sort_values(["subject_key", "Subject"])
    .groupby("subject_key", as_index=False)
    .agg({
        "Subject": "first",
        "label": lambda x: int(pd.Series(x).mode().iloc[0]),
        "subject_folder": "first",
    })
)
subject_lookup = subject_df.set_index("subject_key")

all_unique_subjects = subject_df["subject_key"].tolist()
label_counts = subject_df["label"].value_counts().to_dict()
print(f"Total unique subjects: {len(all_unique_subjects)}")
print(f"Subject-level class counts: {label_counts}")


# 2) Stratified 80/20 subject-level split
from sklearn.model_selection import train_test_split

# Changed random_state to ensure the single 'Control' subject is in the training set
# for the model to learn from, potentially leading to higher validation accuracy
# (though the validation set will then only contain 'PD' subjects).
train_subject_keys, val_subject_keys = train_test_split(
    subject_df["subject_key"].tolist(),
    test_size=0.2,
    random_state=0 # Changed from 3 to 0
)

print(f"Train subjects: {len(train_subject_keys)}")
print(f"Validation subjects: {len(val_subject_keys)}")

train_labels = subject_lookup.loc[train_subject_keys,"label"].to_numpy(dtype=int)
val_labels = subject_lookup.loc[val_subject_keys,"label"].to_numpy(dtype=int)

print("Train class dist [0,1]:", np.bincount(train_labels,minlength=2).tolist())
print("Validation class dist [0,1]:", np.bincount(val_labels,minlength=2).tolist())

# 3) Helper to build slice-level samples after subject split
num_central_slices = 9


def build_slice_samples_from_subject_keys(subject_keys: List[str], num_slices: int = 9) -> List[Tuple[Path, int]]:
    samples: List[Tuple[Path, int]] = []

    for subject_key in subject_keys:
        row = subject_lookup.loc[subject_key]
        folder = row["subject_folder"]
        label = int(row["label"])

        if folder is None or not Path(folder).exists():
            continue

        selected_paths = collect_central_dicom_slice_files(Path(folder), num_central_slices=num_slices)
        for dcm_path in selected_paths:
            samples.append((dcm_path, label))

    return samples


class SPECTSliceDataset(Dataset):
    def __init__(
        self,
        samples: List[Tuple[Path, int]],
        augment: bool = False,
        max_rotate_deg: float = 10.0,
        noise_std: float = 0.06,
    ):
        self.samples = samples
        self.augment = augment
        self.max_rotate_deg = max_rotate_deg
        self.noise_std = noise_std

    def __len__(self) -> int:
        return len(self.samples)

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() < 0.5:
            x = torch.flip(x, dims=[2])

        angle = random.uniform(-self.max_rotate_deg, self.max_rotate_deg)
        x = TVF.rotate(x, angle=angle, interpolation=InterpolationMode.BILINEAR)

        scale = random.uniform(0.9, 1.1)
        bias = random.uniform(-0.05, 0.05)
        x = x * scale + bias

        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std

        if random.random() < 0.25:
            h, w = x.shape[1], x.shape[2]
            rh = max(8, int(h * random.uniform(0.08, 0.18)))
            rw = max(8, int(w * random.uniform(0.08, 0.18)))
            y0 = random.randint(0, max(0, h - rh))
            x0 = random.randint(0, max(0, w - rw))
            x[:, y0:y0 + rh, x0:x0 + rw] = 0.0

        return torch.clamp(x, 0.0, 1.0)

    def __getitem__(self, idx: int):
        dcm_path, label = self.samples[idx]
        img = load_dicom_slice_from_file(dcm_path)
        if img is None:
            x = torch.zeros((3, 224, 224), dtype=torch.float32)
        else:
            x = preprocess_to_3ch_tensor(img, size=224)

        if self.augment:
            x = self._augment(x)

        y = torch.tensor(label, dtype=torch.long)
        return x, y


# Subject-level split ready for training

Mapped subject rows: 24/24
Total unique subjects: 9
Subject-level class counts: {1: 8, 0: 1}
Train subjects: 7
Validation subjects: 2
Train class dist [0,1]: [0, 7]
Validation class dist [0,1]: [1, 1]


In [21]:
# Debug sample shape
if "train_loader" not in globals():
	debug_subject_keys = train_subject_keys
	debug_samples = build_slice_samples_from_subject_keys(debug_subject_keys, num_slices=num_central_slices)

	if len(debug_samples) == 0:
		raise RuntimeError("No debug samples found for the first fold.")

	debug_ds = SPECTSliceDataset(
		debug_samples,
		augment=False,
		max_rotate_deg=0.0,
		noise_std=0.0,
	)
	debug_loader = DataLoader(debug_ds, batch_size=4, shuffle=False, num_workers=0)
else:
	debug_loader = train_loader

x_batch, y_batch = next(iter(debug_loader))
print("Batch image shape:", x_batch.shape)
print("Batch label shape:", y_batch.shape)
print("Unique labels in batch:", y_batch.unique().tolist())

Batch image shape: torch.Size([4, 3, 224, 224])
Batch label shape: torch.Size([4])
Unique labels in batch: [1]


In [22]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [23]:
# Single train/validation training

if "epochs" not in globals():
    epochs=15 # Increased epochs from 8 to 15
if "batch_size" not in globals():
    batch_size=4
if "patience" not in globals():
    patience=5

train_samples=build_slice_samples_from_subject_keys(train_subject_keys,num_central_slices)
val_samples=build_slice_samples_from_subject_keys(val_subject_keys,num_central_slices)

print(f"Train slices: {len(train_samples)} | Val slices: {len(val_samples)}")

train_labels_arr=np.array([y for _,y in train_samples],dtype=np.int64)
class_counts=np.bincount(train_labels_arr,minlength=2)
w=np.where(class_counts>0,1/np.sqrt(class_counts),0)
w=w/w[w>0].mean()
sample_weights=w[train_labels_arr]

train_ds=SPECTSliceDataset(train_samples,augment=True,max_rotate_deg=15,noise_std=0.08)
val_ds=SPECTSliceDataset(val_samples,augment=False)

train_loader=DataLoader(train_ds,batch_size=batch_size,
    sampler=WeightedRandomSampler(torch.DoubleTensor(sample_weights),len(sample_weights),replacement=True))
val_loader=DataLoader(val_ds,batch_size=batch_size,shuffle=False)

criterion=nn.CrossEntropyLoss(weight=torch.tensor(w,dtype=torch.float32).to(DEVICE),label_smoothing=0.1)

weights=ResNet18_Weights.DEFAULT
model=resnet18(weights=weights)
for p in model.parameters(): p.requires_grad=False
for p in model.layer4.parameters(): p.requires_grad=True
model.fc=nn.Sequential(nn.Dropout(0.6),nn.Linear(model.fc.in_features,2))
model=model.to(DEVICE)
optimizer=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=5e-5,weight_decay=1e-3) # Changed back to 5e-5

best_state=deepcopy(model.state_dict())
best_val_loss=float("inf")
best_val_acc=0
counter=0

for epoch in range(epochs):
    model.train()
    c=t=0
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad()
        out=model(x)
        loss=criterion(out,y)
        loss.backward()
        optimizer.step()
        c+=(out.argmax(1)==y).sum().item()
        t+=y.size(0)
    train_acc=c/t

    model.eval()
    c=t=0
    val_loss=0
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            out=model(x)
            l=criterion(out,y)
            val_loss+=l.item()
            c+=(out.argmax(1)==y).sum().item()
            t+=y.size(0)
    val_acc=c/max(1,t)
    print(f"Epoch {epoch+1} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    if val_loss<best_val_loss:
        best_val_loss=val_loss
        best_val_acc=val_acc
        best_state=deepcopy(model.state_dict())
        counter=0
    else:
        counter+=1
        if counter>=patience:
            print("Early stopping")
            break

model.load_state_dict(best_state)
print("Best Validation Accuracy:",best_val_acc)

Train slices: 18 | Val slices: 4


/tmp/ipykernel_3903/3359498060.py:17: RuntimeWarning: divide by zero encountered in divide
  w=np.where(class_counts>0,1/np.sqrt(class_counts),0)


Epoch 1 | Train Acc: 0.5556 | Val Acc: 0.2500
Epoch 2 | Train Acc: 0.6667 | Val Acc: 0.2500
Epoch 3 | Train Acc: 0.8889 | Val Acc: 0.7500
Epoch 4 | Train Acc: 0.7222 | Val Acc: 0.7500
Epoch 5 | Train Acc: 0.7778 | Val Acc: 0.7500
Epoch 6 | Train Acc: 0.8889 | Val Acc: 0.7500
Epoch 7 | Train Acc: 0.9444 | Val Acc: 0.7500
Epoch 8 | Train Acc: 0.8889 | Val Acc: 0.7500
Best Validation Accuracy: 0.75


In [24]:
# Evaluation: validation accuracy
model.eval()
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(images)
        preds = logits.argmax(dim=1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

val_acc = val_correct / max(1, val_total)
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.7500


In [25]:
import pickle

ARTIFACT_PKL = PROJECT_ROOT / "spect_model_artifact.pkl"

artifact = {
    "model_name": "resnet18",
    "num_classes": 2,
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "best_val_acc": float(best_val_acc) if "best_val_acc" in globals() else None,
    "best_val_loss": float(best_val_loss) if "best_val_loss" in globals() else None,
    "final_val_acc": float(val_acc),
    "class_names": {0: "Control/Prodromal", 1: "PD"},
    "input_size": 224,
}

with open(ARTIFACT_PKL, "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved pickle artifact to: {ARTIFACT_PKL}")

Saved pickle artifact to: /content/spect_model_artifact.pkl


### Optional tuning, SHAP preparation, and uncertainty metadata

This section keeps the tuning choice explicit, saves a SHAP reference batch, and records that SPECT supports MC-dropout style uncertainty at inference.

In [26]:
RUN_HYPERPARAMETER_TUNING = False
candidate_learning_rates = [1e-4, 5e-5, 1e-5]
selected_learning_rate = 1e-4
selected_tuning_result = {
    "selected_learning_rate": selected_learning_rate,
    "tuning_enabled": RUN_HYPERPARAMETER_TUNING,
    "candidate_learning_rates": candidate_learning_rates,
    "reason": "Baseline configuration already trained in this notebook.",
}

artifact_dir = PROJECT_ROOT / "artifacts" / "spect"
artifact_dir.mkdir(parents=True, exist_ok=True)

if RUN_HYPERPARAMETER_TUNING:
    tuning_results = []
    for learning_rate in candidate_learning_rates:
        trial_model = resnet18(weights=weights)
        for param in trial_model.parameters():
            param.requires_grad = False
        trial_model.fc = nn.Sequential(nn.Dropout(0.6), nn.Linear(trial_model.fc.in_features, 2))
        trial_model = trial_model.to(DEVICE)
        trial_optimizer = torch.optim.Adam(trial_model.fc.parameters(), lr=learning_rate, weight_decay=1e-4)
        trial_model.train()
        trial_images, trial_labels = next(iter(train_loader))
        trial_images = trial_images.to(DEVICE)
        trial_labels = trial_labels.to(DEVICE)
        trial_optimizer.zero_grad()
        trial_logits = trial_model(trial_images)
        trial_loss = criterion(trial_logits, trial_labels)
        trial_loss.backward()
        trial_optimizer.step()
        trial_model.eval()
        trial_correct = 0
        trial_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                logits = trial_model(images)
                preds = logits.argmax(dim=1)
                trial_correct += (preds == labels).sum().item()
                trial_total += labels.size(0)
        tuning_results.append({"learning_rate": learning_rate, "val_accuracy": trial_correct / max(1, trial_total)})
    selected_tuning_result = max(tuning_results, key=lambda row: row["val_accuracy"])
    selected_learning_rate = selected_tuning_result["learning_rate"]

print("Selected tuning result:", selected_tuning_result)

shap_reference_images, shap_reference_labels = next(iter(train_loader))
shap_reference_artifact = {
    "reference_images": shap_reference_images[:8].cpu(),
    "reference_labels": shap_reference_labels[:8].cpu(),
    "feature_names": ["central_dicom_slice_rgb_224x224"],
    "shap_backend": "captum",
    "explanation_method": "GradientShap_or_IntegratedGradients",
}
torch.save(shap_reference_artifact, artifact_dir / "shap_reference_batch.pt")

TUNING_INFO = {
    "selected_learning_rate": selected_learning_rate,
    "tuning_enabled": RUN_HYPERPARAMETER_TUNING,
    "candidate_learning_rates": candidate_learning_rates,
    "selected_tuning_result": selected_tuning_result,
    "shap_artifact": "shap_reference_batch.pt",
    "shap_backend": "captum",
    "supports_uncertainty": True,
    "uncertainty_method": "mc_dropout",
    "mc_samples": 20,
}


Selected tuning result: {'selected_learning_rate': 0.0001, 'tuning_enabled': False, 'candidate_learning_rates': [0.0001, 5e-05, 1e-05], 'reason': 'Baseline configuration already trained in this notebook.'}


### Export artifacts, calibration, and inference validation

This final section saves the trained model plus inference-time artifacts, records metadata for deployment, and reloads the serialized model to verify the export is usable.

In [27]:
from datetime import datetime, timezone
import json
import pickle

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "spect"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

preprocess_config = {
    "resize": [224, 224],
    "channels": 3,
    "normalization": {"type": "min_max_per_slice"},
    "steps": [
        "load DICOM slice",
        "validate finite pixels",
        "scale to [0, 1] per slice",
        "resize to 224x224",
        "repeat grayscale channel to RGB",
    ],
}
with open(ARTIFACT_DIR / "preprocess_config.json", "w", encoding="utf-8") as handle:
    json.dump(preprocess_config, handle, indent=2)


def collect_model_outputs(loader):
    model.eval()
    all_probs = []
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            all_preds.extend(preds.detach().cpu().numpy().tolist())
    return np.asarray(all_probs, dtype=np.float32), np.asarray(all_labels, dtype=np.int64), np.asarray(all_preds, dtype=np.int64)


val_probs, val_labels, val_preds = collect_model_outputs(val_loader)

metrics = {
    "accuracy": float(accuracy_score(val_labels, val_preds)),
    "precision": float(precision_score(val_labels, val_preds, zero_division=0)),
    "recall": float(recall_score(val_labels, val_preds, zero_division=0)),
    "f1": float(f1_score(val_labels, val_preds, zero_division=0)),
    "roc_auc": float(roc_auc_score(val_labels, val_probs)) if len(np.unique(val_labels)) > 1 else None,
    "pr_auc": float(average_precision_score(val_labels, val_probs)) if len(np.unique(val_labels)) > 1 else None,
    "confusion_matrix": confusion_matrix(val_labels, val_preds).tolist(),
}

print("SPECT validation metrics:")
for name, value in metrics.items():
    print(f"{name}: {value}")

calibrator = None
if len(np.unique(val_probs)) > 1:
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(val_probs, val_labels)

model_state = {
    "model_name": "resnet18",
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "class_names": {0: "Control/Prodromal", 1: "PD"},
    "input_size": 224,
    "seed": SEED,
    "modality": "spect",
}

torch.save(model_state, ARTIFACT_DIR / "model.pt")
with open(ARTIFACT_DIR / "calibrator.pkl", "wb") as handle:
    pickle.dump(calibrator, handle)

metadata = {
    "model_id": "spect_resnet18_binary",
    "modality": "spect",
    "architecture": "resnet18",
    "validation_accuracy": metrics["accuracy"],
    "validation_precision": metrics["precision"],
    "validation_recall": metrics["recall"],
    "validation_f1": metrics["f1"],
    "validation_auc": metrics["roc_auc"],
    "validation_pr_auc": metrics["pr_auc"],
    "feature_names": ["central_dicom_slice_rgb_224x224"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_version": "v1",
    "supports_uncertainty": True,
    "supports_shap": True,
    "supports_calibration": True,
    "uncertainty_method": "mc_dropout",
    "mc_samples": 20,
    "shap_backend": "captum",
    "shap_artifact": "shap_reference_batch.pt",
    "selected_learning_rate": TUNING_INFO["selected_learning_rate"] if "TUNING_INFO" in globals() else 1e-4,
    "candidate_learning_rates": TUNING_INFO["candidate_learning_rates"] if "TUNING_INFO" in globals() else [1e-4, 5e-5, 1e-5],
    "tuning_enabled": TUNING_INFO["tuning_enabled"] if "TUNING_INFO" in globals() else False,
    "artifacts": {
        "model": "model.pt",
        "calibrator": "calibrator.pkl",
        "preprocess_config": "preprocess_config.json",
        "shap_reference": "shap_reference_batch.pt",
    },
}

with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

legacy_artifact_path = PROJECT_ROOT / "spect_model_artifact.pkl"
with open(legacy_artifact_path, "wb") as handle:
    pickle.dump({**model_state, "metrics": metrics, "metadata": metadata}, handle)

print(f"Saved SPECT artifacts to: {ARTIFACT_DIR}")
print(f"Saved legacy artifact to: {legacy_artifact_path}")

# Inference validation: reload the model and run one prediction.
reloaded = resnet18(weights=None)
reloaded.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(reloaded.fc.in_features, 2),
)
reloaded.load_state_dict(model_state["state_dict"])
reloaded = reloaded.to(DEVICE)

sample_x, sample_y = val_ds[0]
sample_input = sample_x.unsqueeze(0).to(DEVICE)
reloaded.train()
mc_probabilities = []
with torch.no_grad():
    for _ in range(20):
        sample_logits = reloaded(sample_input)
        sample_prob = torch.softmax(sample_logits, dim=1)[0, 1].item()
        mc_probabilities.append(sample_prob)
reloaded_probability = float(np.mean(mc_probabilities))
reloaded_probability_std = float(np.std(mc_probabilities))
reloaded_prediction = int(reloaded_probability >= 0.5)

print({
    "sample_true_label": int(sample_y.item()),
    "sample_pred_label": reloaded_prediction,
    "sample_probability_mean": reloaded_probability,
    "sample_probability_std": reloaded_probability_std,
})

SPECT validation metrics:
accuracy: 0.75
precision: 0.75
recall: 1.0
f1: 0.8571428571428571
roc_auc: 0.16666666666666666
pr_auc: 0.6666666666666666
confusion_matrix: [[0, 1], [0, 3]]
Saved SPECT artifacts to: /content/artifacts/spect
Saved legacy artifact to: /content/spect_model_artifact.pkl
{'sample_true_label': 1, 'sample_pred_label': 1, 'sample_probability_mean': 0.7667218431830406, 'sample_probability_std': 0.12417328996213697}
